In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import json

import sys

sys.path.append("../")

##################################################################
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "3,4,5,6"
##################################################################

import logging
from src.utils import logging_utils
from src.utils import env_utils
import torch
import transformers
from rich.console import Console
from rich.theme import Theme

custom_theme = Theme({"default": "white"})
console = Console(theme=custom_theme)


# Override print globally
def rich_print(*args, **kwargs):
    console.print(*args, **kwargs)


logger = logging.getLogger(__name__)

logging.basicConfig(
    level=logging.DEBUG,
    format=logging_utils.DEFAULT_FORMAT,
    datefmt=logging_utils.DEFAULT_DATEFMT,
    stream=sys.stdout,
)

logger.info(f"{torch.__version__=}, {torch.version.cuda=}")
logger.info(
    f"{torch.cuda.is_available()=}, {torch.cuda.device_count()=}, {torch.cuda.get_device_name()=}"
)
logger.info(f"{transformers.__version__=}")

2026-02-26 11:17:44 __main__ INFO     torch.__version__='2.10.0+cu128', torch.version.cuda='12.8'
2026-02-26 11:17:44 __main__ INFO     torch.cuda.is_available()=True, torch.cuda.device_count()=4, torch.cuda.get_device_name()='NVIDIA A100 80GB PCIe'
2026-02-26 11:17:44 __main__ INFO     transformers.__version__='5.1.0'


## Loading the LM

In [3]:
from src.models import ModelandTokenizer

# model_key = "allenai/Olmo-3-1125-32B"
# model_key = "allenai/Olmo-3.1-32B-Instruct"
# model_key = "allenai/Olmo-3.1-7B-RL-Zero-Code"

# model_key = "meta-llama/Llama-3.3-70B-Instruct"

# model_key = "deepseek-ai/deepseek-coder-33b-base"
# model_key = "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct"

# model_key = "openai/gpt-oss-20b"
# model_key = "openai/gpt-oss-120b" #! weird results. known issue.

# model_key = "Qwen/Qwen3-Coder-30B-A3B-Instruct"
# model_key = "Qwen/Qwen2.5-Coder-14B"
# model_key = "Qwen/Qwen2.5-Coder-14B-Instruct"
# model_key = "Qwen/Qwen2.5-Coder-32B"
model_key = "Qwen/Qwen2.5-Coder-32B-Instruct"
# model_key = "Qwen/Qwen2.5-72B"
# model_key = "Qwen/Qwen2.5-72B-Instruct"

# model_key = "bigcode/starcoder2-15b"

# model_key = "zai-org/GLM-4.7-Flash"

mt = ModelandTokenizer(
    model_key=model_key,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)

2026-02-26 11:17:48 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory
2026-02-26 11:17:48 httpcore.connection DEBUG    connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-02-26 11:17:48 httpcore.connection DEBUG    connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7fe9dfee9790>
2026-02-26 11:17:48 httpcore.connection DEBUG    start_tls.started ssl_context=<ssl.SSLContext object at 0x7fe9e02f3150> server_hostname='huggingface.co' timeout=10
2026-02-26 11:17:48 httpcore.connection DEBUG    start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7fe9dfbbfb00>
2026-02-26 11:17:48 httpcore.http11 DEBUG    send_request_headers.started request=<Request [b'HEAD']>
2026-02-26 11:17:48 httpcore.http11 DEBUG    send_request_headers.complete
2026-02-26 11:17:48 http

Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

2026-02-26 11:18:11 httpcore.connection DEBUG    close.started
2026-02-26 11:18:11 httpcore.connection DEBUG    close.complete
2026-02-26 11:18:11 httpcore.connection DEBUG    connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-02-26 11:18:11 httpcore.connection DEBUG    connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7fe9e0a4f320>
2026-02-26 11:18:11 httpcore.connection DEBUG    start_tls.started ssl_context=<ssl.SSLContext object at 0x7fe9e02f3150> server_hostname='huggingface.co' timeout=10
2026-02-26 11:18:11 httpcore.connection DEBUG    start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7fe9dfee96a0>
2026-02-26 11:18:11 httpcore.http11 DEBUG    send_request_headers.started request=<Request [b'HEAD']>
2026-02-26 11:18:11 httpcore.http11 DEBUG    send_request_headers.complete
2026-02-26 11:18:11 httpcore.http11 DEBUG    send_request_body.started request=<Request [b'HE

## Debug

In [ ]:
from src.goals.eval import run_evaluation, pretty_print_eval_results
from src.goals.predicate_data import MultiPredicateDataset


PREDICATE_IDX = 5

eval_dataset = MultiPredicateDataset.load_dataset()

target_predicate = eval_dataset[PREDICATE_IDX]
rich_print(target_predicate)

In [ ]:
from src.tokens import prepare_input
from src.goals.data import CodeGenerations, Code
from src.goals.functional import supports_chat_template, apply_chat_template
from src.functional import generate_with_patch

SAMPLE_IDX = 2
sample = target_predicate.samples[SAMPLE_IDX]
instruction = sample.instruction
unfinished_code = sample.code
function_name = sample.function_name or "func"
unit_tests = sample.unit_tests
prefills = []
for ptype, pfill_list in sample.prefills.items():
    prefills.extend([prefill, ptype] for prefill in pfill_list)


if supports_chat_template(mt):
    prompts = [
        apply_chat_template(
            mt=mt,
            user_query=instruction.strip(),
            incomplete_code=unfinished_code + prefill,
        )
        for prefill, ptype in prefills
    ]
else:
    prompts = [
        instruction + "<code>" + unfinished_code + prefill
        for (prefill, ptype) in prefills
    ]


inputs = prepare_input(
    prompts=prompts,
    tokenizer=mt,
)

output = generate_with_patch(
    mt=mt,
    inputs=inputs,
    n_gen_per_prompt=1,
    max_new_tokens=64,
    do_sample=False,
    remove_prefix=True,
)

results = CodeGenerations(
    sample=sample,
    generations=[
        Code(prefill=prefill, prefill_type=ptype, full_prompt=inp, lm_generation=out)
        for (prefill, ptype), inp, out in zip(prefills, prompts, output)
    ],
)

# print(f"[white]{str(results)}[/white]")

eval_results = run_evaluation(
    result = results,
    function_name=function_name,
    code_stop_tag="</code>",
)

# Print results in tabular format
pretty_print_eval_results(eval_results)

In [ ]:
prefill_idx = -1
generated_code = eval_results.generations[prefill_idx]
print(generated_code)

print(f'Full Prompt:\n"{generated_code.full_prompt}"\n')
print(f'Full Output:\n"{generated_code.lm_generation}"\n')

In [ ]:
print(target_predicate.samples[2])

In [ ]:
from src.goals.sampling import MCTSSearch

full_prompt = prompts[prefill_idx]
print(f'Full Prompt for MCTS:\n"{full_prompt}"\n')

search = MCTSSearch(
    mt=mt,
    prompt=full_prompt,
    max_depth=32, top_k=5,
    prob_threshold=0.01,
    temperature=1.0,
    c_puct=1.0,
    min_children_up_to_depth=3,
    min_children=2,
)

root = search.run(
    n_iterations=1500,
    depth_first=True,
)

In [ ]:
print(search.get_stats())

paths = search.get_all_paths(
    # terminal_only=True, 
    sort_by="prob", 
    # contains="else False"
    # depth=5
)

# Print results
for p in paths:
    print(
        f'<v={p["visits"]}, p={p["prob"]:.4f}> {repr(p["text"])}'
    )

In [ ]:
# Inspect results
search.print_tree(max_depth=None, show_colors=False)

In [ ]:
from src.functional import predict_next_token

prefix = sample.code + generated_code.prefill
print(prefix)

test = ' == n % 2)'

test_prompt = generated_code.full_prompt + test
print(test_prompt)

predict_next_token(
    mt=mt,
    inputs = test_prompt
)

In [ ]:
# node = search.find_node_by_text(' == n % 2)')
# node

In [ ]:
# # Debug why it's not being expanded
# if node:
#     print(search.debug_node(node))

#     # If it's not terminal, see what children would be created
#     if not node.is_terminal and node.is_leaf():
#         search.force_expand(node)

#     # Check PUCT scores at its parent to see why siblings might be selected first
#     if node.parent:
#         for child, score in search.get_puct_scores(node.parent):
#             print(f"{child.token_str!r}: PUCT={score:.4f}, visits={child.visits}")

In [ ]:
from src.goals.eval import evaluate_mcts_tree, clear_mcts_colors

prefix = sample.code + generated_code.prefill
clear_mcts_colors(search.root)

counts = evaluate_mcts_tree(
    root=search.root,
    prefix=prefix,
    unit_tests=sample.unit_tests,
    function_name=sample.function_name or "func",
    terminal_only=True
)

print(counts)

In [ ]:
search.print_tree(max_depth=None, show_colors=True)

In [ ]:
search.save_tree("tree.html", show_colors=True)

## Checking the green solutions

In [ ]:
# def get_green_solutions(root, prefix, tokenizer=None):                                                                                                                                                                        
#     """Return list of (full_code, eval_result) for all green leaf nodes."""                                                                                                                                                   
#     results = []                                                                                                                                                                                                              
#     def walk(node):                                                                                                                                                                                                           
#         if node.is_leaf() and node.color == "green":                                                                                                                                                                        
#             results.append((prefix + node.get_path_text(), node.eval_result))
#         for child in node.children:
#             walk(child)
#     walk(root)
#     return results

# # After evaluate_mcts_tree(root, prefix, unit_tests, function_name):
# solutions = get_green_solutions(root, prefix="")
# for code, eval_result in solutions:
#     print(code)
#     print("---")

In [ ]:
def get_green_solutions_by_first_token(root, prefix):
    """Group green solutions by their first token (reveals which predicate was pursued first)."""
    from collections import defaultdict
    groups = defaultdict(list)
    def walk(node):
        if node.is_leaf() and node.color == "green":
            # Walk up to find the root's direct child (first token)
            n = node
            while n.parent and not n.parent.is_root():
                n = n.parent
            first_tok = n.token_str
            groups[first_tok].append(prefix + node.get_path_text())
        for child in node.children:
            walk(child)
    walk(root)
    return dict(groups)

groups = get_green_solutions_by_first_token(root, prefix="")
for first_tok, codes in groups.items():
    print(f"First token: {first_tok!r}")
    for code in codes:
        print(f'"{code}"')
        # print("---")

## Build Trees for the Target Predicate

In [43]:
from src.goals.predicate_data import MultiPredicateDataset

PREDICATE_IDX = 2

eval_dataset = MultiPredicateDataset.load_dataset()
target_predicate = eval_dataset[PREDICATE_IDX]
rich_print(target_predicate)

2026-02-26 12:33:43 src.goals.predicate_data INFO     Loaded TargetPredicate 'has_unique_elements' with 3 sample(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/has_unique_elements.json
2026-02-26 12:33:43 src.goals.predicate_data INFO     Loaded TargetPredicate 'is_anagram' with 4 sample(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/is_anagram.json
2026-02-26 12:33:43 src.goals.predicate_data INFO     Loaded TargetPredicate 'is_palindrome' with 5 sample(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/is_palindrome.json
2026-02-26 12:33:43 src.goals.predicate_data INFO     Loaded TargetPredicate 'is_perfect_square' with 4 sample(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/is_perfect_square.json
2026-02-26 12:33:43 src.goals.predicate_data INFO     Loaded TargetPredicate 'is_power_of_two' with 3 sample(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/is_power_of_two.json


2026-02-26 12:33:43 src.goals.predicate_data INFO     Loaded TargetPredicate 'is_sorted' with 4 sample(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/is_sorted.json
2026-02-26 12:33:43 src.goals.predicate_data INFO     Loaded MultiPredicateDataset: 6 target predicate(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate


TargetPredicate(
    p_plus=PredicateSpec(description='check whether s is a palindrome (s == s[::-1])', anchor_tokens=[]),
    samples=[
        MultiPredicateSample(
            instruction='Write a Python function that returns True iff the string has at least 2 characters and is 
a palindrome.\n\n',
            code='def is_valid_palindrome(s: str) -> bool:\n    """Return True iff s has at least 2 characters and 
is a palindrome."""\n    return ',
            unit_tests=[
                ('aba', True),
                ('ab', False),
                ('a', False),
                ('', False),
                ('aa', True),
                ('abcba', True),
                ('abc', False),
                ('racecar', True),
                ('hello', False)
            ],
            prefills={
                'aligned_p_plus': ['s == s', 's[::-1]', 's[:', 's =='],
                'aligned_p_minus': ['len(', 'len(s)'],
                'adversarial': ['not (', 'False if', 'True if not'],
                'inconsequential': ['']
            },
            function_name='is_valid_palindrome',
            p_minus=PredicateSpec(description='check whether len(s) >= 2', anchor_tokens=[])
        ),
        MultiPredicateSample(
            instruction='Write a Python function that returns True iff the string contains only alphabetic 
characters and is a palindrome.\n\n',
            code='def is_alpha_palindrome(s: str) -> bool:\n    """Return True iff s contains only alphabetic 
characters and is a palindrome."""\n    return ',
            unit_tests=[
                ('aba', True),
                ('abc', False),
                ('a1a', False),
                ('121', False),
                ('racecar', True),
                ('hello', False),
                ('', False),
                ('a', True),
                ('ABA', True),
                ('ab1ba', False)
            ],
            prefills={
                'aligned_p_plus': ['s == s', 's[::-1]', 's[:', 's =='],
                'aligned_p_minus': ['s.isalpha()', 's.is'],
                'adversarial': ['not (', 'False if', 'True if not'],
                'inconsequential': ['']
            },
            function_name='is_alpha_palindrome',
            p_minus=PredicateSpec(
                description='check whether s.isalpha() (all characters are alphabetic)',
                anchor_tokens=[]
            )
        ),
        MultiPredicateSample(
            instruction='Write a Python function that returns True iff the string contains no spaces and is a 
palindrome.\n\n',
            code='def is_spaceless_palindrome(s: str) -> bool:\n    """Return True iff s contains no spaces and is 
a palindrome."""\n    return ',
            unit_tests=[
                ('aba', True),
                ('abc', False),
                ('a b a', False),
                ('race car', False),
                ('racecar', True),
                ('hello', False),
                ('a', True),
                ('', True),
                ('a a', False),
                ('abcba', True)
            ],
            prefills={
                'aligned_p_plus': ['s == s', 's[::-1]', 's[:', 's =='],
                'aligned_p_minus': ["' ' not in", '" " not in'],
                'adversarial': ['not (', 'False if', 'True if not'],
                'inconsequential': ['']
            },
            function_name='is_spaceless_palindrome',
            p_minus=PredicateSpec(description="check whether ' ' not in s (no spaces)", anchor_tokens=[])
        ),
        MultiPredicateSample(
            instruction='Write a Python function that returns True iff the first character of the string is 
uppercase and the string is a palindrome.\n\n',
            code='def is_upper_palindrome(s: str) -> bool:\n    """Return True iff s is non-empty, starts with an 
uppercase letter, and is a palindrome."""\n    return ',
            unit_tests=[
                ('AbA', True),
                ('ABA', True),
                ('A

In [44]:
from src.goals.predicate_sampling import (
    MCTSRunResult,
    MCTSSearchResults_for_MultiPredicateSample,
    MCTSSearchResults_for_TargetPredicate,
)
from src.goals.predicate_data import MultiPredicateSample
from src.goals.sampling import MCTSSearch
from src.goals.eval import evaluate_mcts_tree, clear_mcts_colors
from src.goals.anchor_extraction import get_green_completions
from src.goals.functional import supports_chat_template, apply_chat_template


def build_full_prompt(sample: MultiPredicateSample, prefill: str, mt) -> str:
    if supports_chat_template(mt):
        return apply_chat_template(
            mt=mt,
            user_query=sample.instruction.strip(),
            incomplete_code=sample.code + prefill,
        )
    return sample.instruction + "<code>" + sample.code + prefill


MCTS_CONFIG = dict(
    n_iterations=1500,
    max_depth=32,
    top_k=5,
    prob_threshold=0.01,
    temperature=1.0,
    c_puct=1.0,
    min_children_up_to_depth=6,
    min_children=4,
    depth_first=True,
)

In [45]:
import os

target_predicate = eval_dataset[PREDICATE_IDX]
model_tag = mt.name.split("/")[-1]
SAVE_PATH = os.path.join(
    env_utils.DEFAULT_RESULTS_DIR,
    "MultiPredicate_MCTS", 
    model_tag,
    f"{target_predicate.name}.json"
)

os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)

# ── Load cached results if available ─────────────────────────────────────────
if os.path.exists(SAVE_PATH):
    rich_print(f"[yellow]Loading cached results from {SAVE_PATH}[/yellow]")
    tp_results = MCTSSearchResults_for_TargetPredicate.load(SAVE_PATH)

# ── Otherwise run MCTS for every sample × every prefill ──────────────────────
else:
    tp_results = MCTSSearchResults_for_TargetPredicate(
        target_predicate_name=target_predicate.name,
        model_name=mt.name,
        mcts_config=MCTS_CONFIG,
    )

    for sample_idx, p_neg_sample in enumerate(target_predicate.samples):
        rich_print(
            f"\n[bold]Sample {sample_idx + 1}/{len(target_predicate.samples)}[/bold]"
            f"  fn=[cyan]{p_neg_sample.function_name}[/cyan]"
            f"  p-=[italic]{p_neg_sample.p_minus.description if p_neg_sample.p_minus else '?'}[/italic]"
        )

        sample_result = MCTSSearchResults_for_MultiPredicateSample(sample=p_neg_sample)

        all_prefills = [
            (ptype, prefill)
            for ptype, prefills in p_neg_sample.prefills.items()
            for prefill in prefills
        ]

        for prefill_type, prefill in all_prefills:
            # if prefill_type != "inconsequential":
            #     continue
            prompt = build_full_prompt(p_neg_sample, prefill, mt)
            prefix = p_neg_sample.code + prefill

            print(f"  {prefill_type} | prefill={prefill!r}")

            # 1) MCTS search
            search = MCTSSearch(
                mt=mt,
                prompt=prompt,
                max_depth=MCTS_CONFIG["max_depth"],
                top_k=MCTS_CONFIG["top_k"],
                prob_threshold=MCTS_CONFIG["prob_threshold"],
                temperature=MCTS_CONFIG["temperature"],
                c_puct=MCTS_CONFIG["c_puct"],
                min_children_up_to_depth=MCTS_CONFIG["min_children_up_to_depth"],
                min_children=MCTS_CONFIG["min_children"],
            )
            root = search.run(
                n_iterations=MCTS_CONFIG["n_iterations"],
                depth_first=MCTS_CONFIG["depth_first"],
            )

            # 2) Evaluate tree (sets .color and .eval_result on leaves)
            counts = evaluate_mcts_tree(
                root=root,
                prefix=prefix,
                unit_tests=p_neg_sample.unit_tests,
                function_name=p_neg_sample.function_name or "func",
                terminal_only=True,
            )

            # 3) Extract unique green completions
            green_completions = sorted(set(
                c.strip() for c in get_green_completions(root, prefix) if c.strip()
            ))

            run = MCTSRunResult(
                prefill=prefill,
                prefill_type=prefill_type,
                prompt=prompt,
                prefix=prefix,
                mcts_config=MCTS_CONFIG,
                mcts_stats=search.get_stats(),
                eval_counts=counts,
                green_completions=green_completions,
                tree=root.to_dict(),   # serialize the full evaluated tree
            )
            sample_result.runs.append(run)
            sample_result._trees[prefix] = root       # live reference for this session

            rich_print(f"    eval: {counts} | {len(green_completions)} unique green completions")

        tp_results.sample_results.append(sample_result)

    tp_results.save(SAVE_PATH)
    rich_print(f"\n[green]Saved → {SAVE_PATH}[/green]")

# ── Summary ───────────────────────────────────────────────────────────────────
rich_print(f"\n[bold]Summary[/bold] — [cyan]{tp_results.target_predicate_name}[/cyan]  ({tp_results.model_name})")
for i, sr in enumerate(tp_results.sample_results):
    for run in sr.runs:
        rich_print(
            f"  sample {i} | [{run.prefill_type}] prefill={run.prefill!r:10s}"
            f" | {run.eval_counts} | {len(run.green_completions)} green"
        )

Sample 1/5  fn=is_valid_palindrome  p-=check whether len(s) >= 2

  aligned_p_plus | prefill='s == s'
2026-02-26 12:34:01 src.goals.sampling DEBUG    MCTS iteration 50/1500, total nodes: 185
2026-02-26 12:34:01 src.goals.sampling DEBUG    MCTS iteration 100/1500, total nodes: 185
2026-02-26 12:34:01 src.goals.sampling DEBUG    MCTS iteration 150/1500, total nodes: 185
2026-02-26 12:34:01 src.goals.sampling DEBUG    MCTS iteration 200/1500, total nodes: 185
2026-02-26 12:34:01 src.goals.sampling DEBUG    MCTS iteration 250/1500, total nodes: 185
2026-02-26 12:34:01 src.goals.sampling DEBUG    MCTS iteration 300/1500, total nodes: 185
2026-02-26 12:34:01 src.goals.sampling DEBUG    MCTS iteration 350/1500, total nodes: 185
2026-02-26 12:34:01 src.goals.sampling DEBUG    MCTS iteration 400/1500, total nodes: 185
2026-02-26 12:34:01 src.goals.sampling DEBUG    MCTS iteration 450/1500, total nodes: 185
2026-02-26 12:34:01 src.goals.sampling DEBUG    MCTS iteration 500/1500, total nodes: 185
2026-02-26 12:34:01 src.goals.sampling DEBUG    MCTS iteration 55

eval: {'green': 10, 'orange': 9, 'red': 3, 'total': 22} | 5 unique green completions

  aligned_p_plus | prefill='s[::-1]'
2026-02-26 12:34:10 src.goals.sampling DEBUG    MCTS iteration 50/1500, total nodes: 208
2026-02-26 12:34:10 src.goals.sampling DEBUG    MCTS iteration 100/1500, total nodes: 208
2026-02-26 12:34:10 src.goals.sampling DEBUG    MCTS iteration 150/1500, total nodes: 208
2026-02-26 12:34:10 src.goals.sampling DEBUG    MCTS iteration 200/1500, total nodes: 208
2026-02-26 12:34:10 src.goals.sampling DEBUG    MCTS iteration 250/1500, total nodes: 212
2026-02-26 12:34:10 src.goals.sampling DEBUG    MCTS iteration 300/1500, total nodes: 212
2026-02-26 12:34:10 src.goals.sampling DEBUG    MCTS iteration 350/1500, total nodes: 212
2026-02-26 12:34:10 src.goals.sampling DEBUG    MCTS iteration 400/1500, total nodes: 212
2026-02-26 12:34:10 src.goals.sampling DEBUG    MCTS iteration 450/1500, total nodes: 212
2026-02-26 12:34:10 src.goals.sampling DEBUG    MCTS iteration 500/1500, total nodes: 212
2026-02-26 12:34:10 src.goals.sampling DEBUG    MCTS iteration 5

eval: {'green': 18, 'orange': 0, 'red': 4, 'total': 22} | 9 unique green completions

  aligned_p_plus | prefill='s[:'
2026-02-26 12:34:47 src.goals.sampling DEBUG    MCTS iteration 50/1500, total nodes: 671
2026-02-26 12:34:54 src.goals.sampling DEBUG    MCTS iteration 100/1500, total nodes: 797
2026-02-26 12:34:54 src.goals.sampling DEBUG    MCTS iteration 150/1500, total nodes: 797
2026-02-26 12:34:54 src.goals.sampling DEBUG    MCTS iteration 200/1500, total nodes: 797
2026-02-26 12:35:04 src.goals.sampling DEBUG    MCTS iteration 250/1500, total nodes: 988
2026-02-26 12:35:04 src.goals.sampling DEBUG    MCTS iteration 300/1500, total nodes: 988
2026-02-26 12:35:04 src.goals.sampling DEBUG    MCTS iteration 350/1500, total nodes: 988
2026-02-26 12:35:04 src.goals.sampling DEBUG    MCTS iteration 400/1500, total nodes: 988
2026-02-26 12:35:04 src.goals.sampling DEBUG    MCTS iteration 450/1500, total nodes: 988
2026-02-26 12:35:06 src.goals.sampling DEBUG    MCTS iteration 500/1500, total nodes: 1013
2026-02-26 12:35:06 src.goals.sampling DEBUG    MCTS iteration 550/

... (154 outputs left out)

sample 4 |  prefill='not ('    | {'green': 178, 'orange': 91, 'red': 18, 'total': 287} | 89 green

sample 4 |  prefill='False if' | {'green': 34, 'orange': 16, 'red': 0, 'total': 50} | 17 green

sample 4 |  prefill='True if not' | {'green': 64, 'orange': 27, 'red': 1, 'total': 92} | 32 green

sample 4 |  prefill=''         | {'green': 18, 'orange': 17, 'red': 1, 'total': 36} | 9 green

## Check a Tree to make sure it looks right

In [38]:
SAMPLE_IDX = 0
rich_print(target_predicate.samples[SAMPLE_IDX])

MultiPredicateSample(
    instruction='Write a Python function that returns True iff the list has at least 2 elements and contains no 
duplicate values.\n\n',
    code='def is_long_unique(lst: list) -> bool:\n    """Return True iff lst has at least 2 elements and contains 
no duplicate values."""\n    return ',
    unit_tests=[
        ([1, 2, 3], True),
        ([1, 1, 2], False),
        ([1], False),
        ([], False),
        ([3, 1, 2], True),
        ([1, 2], True),
        ([5, 5], False),
        ([-1, 0, 1], True),
        ([0, 0], False)
    ],
    prefills={
        'aligned_p_plus': ['set(lst)', 'len(set(', 'len(lst) == len(set'],
        'aligned_p_minus': ['len(', 'len(lst)'],
        'adversarial': ['not (', 'False if', 'True if not'],
        'inconsequential': ['']
    },
    function_name='is_long_unique',
    p_minus=PredicateSpec(description='check whether len(lst) >= 2', anchor_tokens=[])
)

In [39]:
len(sr.runs)

9

In [42]:
from src.goals.sampling import print_tree

PREFILL_IDX = -1

sr   = tp_results.sample_results[SAMPLE_IDX]
run  = sr.runs[PREFILL_IDX]
root = sr._trees.get(run.prefix)

if root is None:
    rich_print("[red]Tree missing — was this result saved without tree=root.to_dict()? Re-run MCTS.[/red]")
else:
    rich_print(
        f"[bold]Tree[/bold]  sample={SAMPLE_IDX}  fn=[cyan]{sr.sample.function_name}[/cyan]"
        f"  prefill={run.prefill!r}  eval={run.eval_counts}"
    )
    print_tree(root, show_colors=True)

Tree  sample=0  fn=is_long_unique  prefill=''  eval={'green': 18, 'orange': 6, 'red': 8, 'total': 32}

[ROOT] visits=1676
    ├── '2' (p=0.941, v=1556)
    │   ├── ' <=' (p=0.996, v=1495)
    │   │   ├── ' len' (p=1.000, v=1494)
    │   │   │   ├── '(lst' (p=1.000, v=1493)
    │   │   │   │   ├── ')' (p=1.000, v=1492)
    │   │   │   │   │   ├── ' ==' (p=0.902, v=1345)
    │   │   │   │   │   │   ├── ' len' (p=1.000, v=1344)
    │   │   │   │   │   │   │   └── '(set' (p=1.000, v=1343)
    │   │   │   │   │   │   │       └── '(lst' (p=1.000, v=1342)
    │   │   │   │   │   │   │           ├── '))\n' (p=0.852, v=1143) [TERMINAL] ('2 <= len(lst) == len(set(lst))\n')
    │   │   │   │   │   │   │           └── '))\n\n' (p=0.147, v=198) [TERMINAL] ('2 <= len(lst) == len(set(lst))\n\n')
    │   │   │   │   │   │   ├── 'len' (p=0.000, v=0) ('2 <= len(lst) ==len')
    │   │   │   │   │   │   ├── ' (' (p=0.000, v=0) ('2 <= len(lst) == (')
    │   │   │   │   │   │   └── ' ' (p=0.000, v=0) ('2 <= len(lst) == ')
    │   │   │   │   │   ├── ' and' (p=0.095, v=142)
    │   │   │   │   │   │   ├── ' len' (p=1.000, v=142)
    │   │   │   │   │   │   │   ├── '(lst' (p=0.836, v=118)
    │   │   │   │   │   │   │   │   └── ')' (p=1.000, v=117)
    │   │   │   │   │   │   │   │       └── ' ==' (p=1.000, v=116)
    │   │   │   │   │   │   │   │           └── ' len' (p=1.000, v=115)
    │   │   │   │   │   │   │   │               └── '(set' (p=1.000, v=114)
    │   │   │   │   │   │   │   │                   └── '(lst' (p=1.000, v=113)
    │   │   │   │   │   │   │   │                       ├── '))\n' (p=0.836, v=94) [TERMINAL] ('2 <= len(lst) and len(lst) == len(set(lst))\n')
    │   │   │   │   │   │   │   │                       └── '))\n\n' (p=0.164, v=18) [TERMINAL] ('2 <= len(lst) and len(lst) == len(set(lst))\n\n')
    │   │   │   │   │   │   │   └── '(set' (p=0.164, v=23)
    │   │   │   │   │   │   │       └── '(lst' (p=1.000, v=23)
    │   │   │   │   │   │   │           └── '))' (p=1.000, v=22)
    │   │   │   │   │   │   │               └── ' ==' (p=1.000, v=21)
    │   │   │   │   │   │   │                   └── ' len' (p=1.000, v=20)
    │   │   │   │   │   │   │                       └── '(lst' (p=1.000, v=19)
    │   │   │   │   │   │   │                           ├── ')\n' (p=0.777, v=14) [TERMINAL] ('2 <= len(lst) and len(set(lst)) == len(lst)\n')
    │   │   │   │   │   │   │                           └── ')\n\n' (p=0.223, v=4) [TERMINAL] ('2 <= len(lst) and len(set(lst)) == len(lst)\n\n')
    │   │   │   │   │   │   ├── ' all' (p=0.001, v=0) ('2 <= len(lst) and all')
    │   │   │   │   │   │   ├── ' (' (p=0.000, v=0) ('2 <= len(lst) and (')
    │   │   │   │   │   │   └── ' ' (p=0.000, v=0) ('2 <= len(lst) and ')
    │   │   │   │   │   ├── ' <=' (p=0.001, v=4)
    │   │   │   │   │   │   ├── ' len' (p=1.000, v=4)
    │   │   │   │   │   │   │   └── '(set' (p=1.000, v=3)
    │   │   │   │   │   │   │       └── '(lst' (p=1.000, v=2)
    │   │   │   │   │   │   │           ├── '))\n' (p=0.773, v=1) [TERMINAL] ('2 <= len(lst) <= len(set(lst))\n')
    │   │   │   │   │   │   │           └── '))\n\n' (p=0.222, v=0) [TERMINAL] ('2 <= len(lst) <= len(set(lst))\n\n')
    │   │   │   │   │   │   ├── ' ' (p=0.000, v=0) ('2 <= len(lst) <= ')
    │   │   │   │   │   │   ├── 'len' (p=0.000, v=0) ('2 <= len(lst) <=len')
    │   │   │   │   │   │   └── ' (' (p=0.000, v=0) ('2 <= len(lst) <= (')
    │   │   │   │   │   └── ' <' (p=0.001, v=0) ('2 <= len(lst) <')
    │   │   │   │   ├── '))' (p=0.000, v=0) ('2 <= len(lst))')
    │   │   │   │   ├── ')\n' (p=0.000, v=0) [TERMINAL] ('2 <= len(lst)\n')
    │   │   │   │   └── '）' (p=0.000, v=0) ('2 <= len(lst）')
    │   │   │   ├── '(set' (p=0.000, v=0) ('2 <= len(set')
    │   │   │   ├── ' lst' (p=0.000, v=0) ('2 <= len lst')
    │   │   │   └── '(ls' (p=0.000, v=0) ('2 <= len(ls')
    │   │   ├── ' (' (p=0.000, v=0) ('2 <= (')
    │   │   ├── '(len' (p=0.000, v=0) ('2 <=(len')
    │   │   └── 'len' (p=0.000, v=0) ('2 <=len')
    │   ├── ' &' (p=0.002, v=60)
    │   │   ├── 'lt' (p=0.996, v=11)


In [41]:
def get_green_solutions_by_first_token(root, prefix):
    """Group green solutions by their first token (reveals which predicate was pursued first)."""
    from collections import defaultdict
    groups = defaultdict(list)
    def walk(node):
        if node.is_leaf() and node.color == "green":
            # Walk up to find the root's direct child (first token)
            n = node
            while n.parent and not n.parent.is_root():
                n = n.parent
            first_tok = n.token_str
            groups[first_tok].append(prefix + node.get_path_text())
        for child in node.children:
            walk(child)
    walk(root)
    return dict(groups)

groups = get_green_solutions_by_first_token(root, prefix="")
for first_tok, codes in groups.items():
    print(f"First token: {first_tok!r}")
    for code in codes:
        print(f'"{repr(code)}"')
        # print("---")

First token: ' =='
"' == set() and len(lst) >= 2 if not lst else len(set(lst)) == len(lst) and len(lst) >= 2\n\n'"
"' == set() and len(lst) >= 2 if not lst else len(set(lst)) == len(lst) and len(lst) >= 2\n'"
"' == set(lst) and len(lst) >= 2 and len(set(lst)) == len(lst)\n'"
"' == set(lst) and len(lst) >= 2 and len(set(lst)) == len(lst)\n\n'"
"' == set(lst) and len(lst) >= 2 and len(lst) == len(set(lst))\n'"
"' == set(lst) and len(lst) >= 2 and len(lst) == len(set(lst))\n\n'"
"' == set(lst) and len(lst) > 1 and len(set(lst)) == len(lst)\n'"
"' == set(lst) and len(lst) > 1 and len(set(lst)) == len(lst)\n\n'"
"' == set(lst) and len(lst) > 1 and len(lst) == len(set(lst))\n'"
"' == set(lst) and len(lst) > 1 and len(lst) == len(set(lst))\n\n'"
"' == set(lst) and len(lst) == len(set(lst)) and len(lst) >= 2\n'"
"' == set(lst) and len(lst) == len(set(lst)) and len(lst) >= 2\n\n'"
"' == set(lst) and len(set(lst)) == len(lst) and len(lst) >= 2\n'"
"' == set(lst) and len(set(lst)) == len(lst) and

## Anchor Extration with Oracle LLMs

In [ ]:
from src.goals.anchor_extraction import extract_anchor_tokens

# ── Run oracle extraction ──────────────────────────────────────────────────
# tp_results.completions_per_sample is list[dict[prefix → [completions]]]
# one entry per sample, matching the order of target_predicate.samples
p_plus_anchors, p_minus_anchors_per_sample = extract_anchor_tokens(
    tp=target_predicate,
    completions_per_sample=tp_results.completions_per_sample,
    use_cache=True,
)

# ── Inspect ────────────────────────────────────────────────────────────────
rich_print(f"\n[bold]P+ anchors[/bold] ([cyan]{target_predicate.p_plus.description}[/cyan]):")
rich_print(p_plus_anchors)

for i, (sample, p_minus_anchors) in enumerate(
    zip(target_predicate.samples, p_minus_anchors_per_sample)
):
    p_minus_desc = sample.p_minus.description if sample.p_minus else "?"
    rich_print(f"\n[bold]P- anchors  sample {i}[/bold] ([magenta]{p_minus_desc}[/magenta]):")
    rich_print(p_minus_anchors)

2026-02-25 18:21:46 src.goals.anchor_extraction DEBUG    [is_sorted] sample=0: both predicates already identified in completion — skipping oracle call.
2026-02-25 18:21:46 src.goals.anchor_extraction DEBUG    [is_sorted] sample=0: both predicates already identified in completion — skipping oracle call.
2026-02-25 18:21:46 src.goals.anchor_extraction DEBUG    [is_sorted] sample=0: both predicates already identified in completion — skipping oracle call.
2026-02-25 18:21:46 src.goals.anchor_extraction DEBUG    [is_sorted] sample=0: both predicates already identified in completion — skipping oracle call.
2026-02-25 18:21:46 src.goals.anchor_extraction DEBUG    [is_sorted] sample=0: both predicates already identified in completion — skipping oracle call.
2026-02-25 18:21:46 src.goals.anchor_extraction DEBUG    [is_sorted] sample=0: both predicates already identified in completion — skipping oracle call.
2026-02-25 18:21:46 src.goals.anchor_extraction DEBUG    [is_sorted] sample=0: both pred

P+ anchors (check whether lst is sorted in non-decreasing order (lst == sorted(lst))):

[
    '< lst[i]',
    '<= lst[',
    '<= lst[i',
    '<= lst[i+1]',
    '== sorted',
    '== sorted(',
    '== sorted(lst)',
    'a < b',
    'a == b',
    'all(',
    'all(i',
    'all(lst',
    'all(x',
    'earlier < later',
    'i + 1]',
    'i < j',
    'i <= j',
    'i == j',
    'i+1]',
    'lst ==',
    'lst == sorted',
    'lst[1:]',
    'lst[:-1]',
    'lst[i +',
    'lst[i + 1]',
    'lst[i+1]',
    'lst[i-1] < lst[i]',
    'lst[i-1] <=',
    'lst[i-1] <= lst[i]',
    'lst[i]',
    'lst[i] <',
    'lst[i] <=',
    'lst[i] <= lst[i+1]',
    'sorted(',
    'sorted(lst)',
    'x < y',
    'x <= y',
    'x == y',
    'zip(lst'
]

P- anchors  sample 0 (check whether len(lst) >= 2):

[
    '1 <',
    '1 < len',
    '2 <=',
    '< len(',
    '<= len(',
    '> 1',
    '>= 2',
    'len(',
    'len(lst)',
    'len(lst) >',
    'len(lst) > 1',
    'len(lst) >=',
    'len(lst) >= 2'
]

P- anchors  sample 1 (check whether lst is non-empty and lst[0] >= 0):

['>= 0', 'bool(lst)', 'len(lst)', 'lst) and lst[0]', 'lst[0]', 'lst[0] >=', 'lst[0] >= 0']

P- anchors  sample 2 (check whether len(set(lst)) == len(lst) (all elements unique)):

['== len', '== len(lst', '== len(lst)', 'len(lst) ==', 'len(set', 'len(set(', 'set(', 'set(lst)']

P- anchors  sample 3 (check whether all(isinstance(x, int) for x in lst)):

['int) for', 'isinstance', 'isinstance(i, int)', 'isinstance(x, int)']

In [ ]:
# ── Set anchors on target_predicate and save to disk ──────────────────────
# Review the printed anchors above, edit the lists if needed, then run this cell.

target_predicate.p_plus.anchor_tokens = p_plus_anchors

for i, sample in enumerate(target_predicate.samples):
    if sample.p_minus and p_minus_anchors_per_sample[i]:
        sample.p_minus.anchor_tokens = p_minus_anchors_per_sample[i]

target_predicate.save()   # overwrites data/multi_predicate/<name>.json

rich_print(f"[green]Saved updated anchor tokens → {target_predicate._source_path}[/green]")

2026-02-25 18:22:27 src.goals.predicate_data INFO     Saved TargetPredicate 'is_sorted' to /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/is_sorted.json


Saved updated anchor tokens → /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/is_sorted.json

## Checking contrasting sampling

In [ ]:
from src.goals.probe import (
    compute_outcome_probs,
    sample_contrastive_pairs,
    sample_multidepth_pairs,
    print_contrastive_pairs,
    summarize_contrastive_pairs,
)

# Sample contrastive pairs from the evaluated MCTS tree
# The tree should already be evaluated (search.root has colors assigned)
pairs_by_depth = sample_multidepth_pairs(
    root=search.root,
    min_depth=1,
    max_depth=16,
    min_green_confidence=0.5,
    max_red_in_positive=0.3,  # Slightly relaxed
    min_red_confidence=0.5,
    max_green_in_negative=0.1,
    max_uncertain=0.5,
)

print_contrastive_pairs(pairs_by_depth, max_pairs_per_depth=5)

In [ ]:
# Inspect outcome probabilities at a specific depth
depth_to_inspect = 3
from src.goals.probe import collect_nodes_at_depth

nodes = collect_nodes_at_depth(search.root, depth_to_inspect)
print(f"Nodes at depth {depth_to_inspect}: {len(nodes)}")

for node in nodes[:10]:  # Show first 10
    if not node.is_leaf() and not node.is_terminal:
        probs = compute_outcome_probs(node)
        path_text = node.get_path_text()[:40]
        print(f"  '{path_text}...' -> {probs}")

In [ ]:
# Look at a specific contrastive pair in detail
if pairs_by_depth:
    # Get pairs from the first depth that has them
    first_depth = min(pairs_by_depth.keys())
    pairs = pairs_by_depth[first_depth]
    
    if pairs:
        pair = pairs[0]
        print(f"[bold]Example Contrastive Pair at depth {first_depth}:[/bold]")
        print()
        print(f"[green]Positive path:[/green] {repr(pair.positive.get_path_text())}")
        print(f"  Outcome probs: {pair.pos_probs}")
        print(f"  Node has {len(pair.positive.children)} children")
        print()
        print(f"[red]Negative path:[/red] {repr(pair.negative.get_path_text())}")
        print(f"  Outcome probs: {pair.neg_probs}")
        print(f"  Node has {len(pair.negative.children)} children")
else:
    print("[yellow]No contrastive pairs found. Try adjusting thresholds or running more MCTS iterations.[/yellow]")